# 🧪 Auditor BI-RADS con RoBERTa Clínico — Experimento de mejora

Este notebook es un **experimento independiente**: reemplaza BETO (español general) por **`roberta-base-biomedical-clinical-es`**, un modelo preentrenado en texto **clínico** español. La hipótesis es que un modelo del dominio represente mejor los hallazgos radiológicos y mejore la inferencia del BI-RADS desde **solo las observaciones** (sin la conclusión).

- **Experimento controlado:** se cambia únicamente el modelo base; el balanceo (pesos de clase + augmentación de train) se mantiene igual que en el auditor BETO, para aislar el efecto del modelo.
- **Sin fuga de datos:** la augmentación se aplica solo al conjunto de entrenamiento.
- Al final se compara contra el auditor BETO (F1-macro honesto = 0,4773).

## 1 · Configuración y librerías

In [1]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import random
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, classification_report
from collections import Counter

# Módulo de limpieza compartido (para inferencia, si se requiere)
import sys
sys.path.append('..')
from src.preprocessing import limpiar_texto, MAX_LENGTH

# Reproducibilidad
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# Dispositivo (cuda -> mps -> cpu)
if torch.cuda.is_available():        DEVICE = 'cuda'
elif torch.backends.mps.is_available(): DEVICE = 'mps'
else:                                 DEVICE = 'cpu'

# Modelo base clínico (BSC/PlanTL). Si el primero falla, usa el alterno.
MODELO     = 'PlanTL-GOB-ES/roberta-base-biomedical-clinical-es'
MODELO_ALT = 'BSC-LT/roberta-base-biomedical-clinical-es'

print(f"PyTorch: {torch.__version__} | Dispositivo: {DEVICE}")
print(f"Modelo base: {MODELO}")

PyTorch: 2.12.0 | Dispositivo: mps
Modelo base: PlanTL-GOB-ES/roberta-base-biomedical-clinical-es


## 2 · Carga del dataset curado

In [2]:
# Mismo dataset curado que el notebook principal (salida de la curaduría).
df = pd.read_csv('../data/processed/dataset_clean.csv', encoding='utf-8')
print(f"Informes: {len(df)} | Columnas: {df.shape[1]}")
print(f"Distribución BI-RADS:")
print(df['BI-RADS'].value_counts().sort_index().to_string())

Informes: 4357 | Columnas: 19
Distribución BI-RADS:
BI-RADS
0     966
1     596
2    2635
3      87
4      52
5      16
6       5


## 3 · División train / validación / prueba (solo observaciones)

El auditor usa **únicamente las observaciones** (`auditor_input`), sin la conclusión. División estratificada 70/15/15, semilla 42 — idéntica a la del auditor BETO para comparar de igual a igual.

In [3]:
X = df['auditor_input'].values
y = df['BI-RADS'].values

X_tv, X_test, y_tv, y_test = train_test_split(
    X, y, test_size=0.15, random_state=SEED, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(
    X_tv, y_tv, test_size=0.176, random_state=SEED, stratify=y_tv)

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

Train: 3051 | Val: 652 | Test: 654


## 4 · Pesos de clase

Se calculan **solo sobre el conjunto de entrenamiento** para compensar el desbalance.

In [4]:
clases = np.unique(y_train)
pesos = compute_class_weight(class_weight='balanced', classes=clases, y=y_train)
# Vector de 7 posiciones (BI-RADS 0..6); clases ausentes en train -> peso 1.0
peso_vec = np.ones(7, dtype=np.float32)
for c, w in zip(clases, pesos):
    peso_vec[c] = w
peso_tensor = torch.tensor(peso_vec, dtype=torch.float32).to(DEVICE)
print("Pesos por clase (0..6):", np.round(peso_vec, 2))

Pesos por clase (0..6): [  0.64   1.04   0.24   7.15  12.11  36.32 145.29]


## 5 · Augmentación de clases minoritarias (SOLO train)

Genera variantes sintéticas de las clases de riesgo (BI-RADS 3–6) **únicamente del conjunto de entrenamiento**, para no filtrar información de validación/prueba.

In [5]:
def aumentar_texto(texto):
    variaciones = [
        ('bordes irregulares', 'márgenes irregulares'),
        ('bordes espiculados', 'márgenes espiculados'),
        ('bordes mal definidos', 'límites imprecisos'),
        ('imagen nodular', 'nódulo'), ('nódulo', 'imagen nodular'),
        ('lesión nodular', 'nódulo'),
        ('heterogéneamente densas', 'de densidad heterogénea'),
        ('muy densas', 'extremadamente densas'),
        ('calcificaciones sospechosas', 'depósitos cálcicos sospechosos'),
        ('microcalcificaciones', 'calcificaciones puntiformes agrupadas'),
        ('mama derecha', 'hemimama derecha'), ('mama izquierda', 'hemimama izquierda'),
        ('se visualiza', 'se observa'), ('se observa', 'se visualiza'),
        ('se evidencia', 'se observa'),
    ]
    t = texto
    for orig, rep in random.sample(variaciones, min(3, len(variaciones))):
        if orig in t:
            t = t.replace(orig, rep, 1)
    return t

mask_min = np.isin(y_train, [3, 4, 5, 6])
X_min, y_min = X_train[mask_min], y_train[mask_min]

textos_aug, labels_aug = [], []
for txt, lab in zip(X_min, y_min):
    for _ in range(3):
        textos_aug.append(aumentar_texto(txt))
        labels_aug.append(lab)

X_train_aug = np.concatenate([X_train, np.array(textos_aug)])
y_train_aug = np.concatenate([y_train, np.array(labels_aug)])
idx = np.random.permutation(len(X_train_aug))
X_train_aug, y_train_aug = X_train_aug[idx], y_train_aug[idx]

print(f"Minoritarios en train: {len(X_min)} -> sintéticos: {len(textos_aug)}")
print(f"Train aumentado: {len(X_train_aug)} (sin fuga de val/test)")

Minoritarios en train: 112 -> sintéticos: 336
Train aumentado: 3387 (sin fuga de val/test)


## 6 · Tokenizador (RoBERTa clínico)

In [6]:
try:
    tokenizer = AutoTokenizer.from_pretrained(MODELO)
except Exception as e:
    print(f"Fallback a {MODELO_ALT} ({e})")
    MODELO = MODELO_ALT
    tokenizer = AutoTokenizer.from_pretrained(MODELO)

ejemplo = tokenizer(str(X_train[0]), truncation=True, max_length=MAX_LENGTH)
print(f"Tokenizador cargado. Ejemplo -> {len(ejemplo['input_ids'])} tokens")
print(f"Tokens especiales: CLS={tokenizer.cls_token} SEP={tokenizer.sep_token}")

config.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.27k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.22M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/540k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

Tokenizador cargado. Ejemplo -> 111 tokens
Tokens especiales: CLS=<s> SEP=</s>


## 7 · Dataset y DataLoader

Nota: RoBERTa no usa `token_type_ids`, así que el Dataset solo entrega `input_ids` y `attention_mask`.

In [7]:
class BIRADSDataset(Dataset):
    def __init__(self, textos, labels, tokenizer, max_length=MAX_LENGTH):
        self.textos = list(textos); self.labels = list(labels)
        self.tok = tokenizer; self.max_length = max_length
    def __len__(self): return len(self.textos)
    def __getitem__(self, i):
        enc = self.tok(str(self.textos[i]), truncation=True, padding='max_length',
                       max_length=self.max_length, return_tensors='pt',
                       return_token_type_ids=False)
        return {'input_ids': enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0),
                'labels': torch.tensor(self.labels[i], dtype=torch.long)}

BATCH = 16
train_loader = DataLoader(BIRADSDataset(X_train_aug, y_train_aug, tokenizer), batch_size=BATCH, shuffle=True)
val_loader   = DataLoader(BIRADSDataset(X_val,  y_val,  tokenizer), batch_size=BATCH)
test_loader  = DataLoader(BIRADSDataset(X_test, y_test, tokenizer), batch_size=BATCH)
print(f"Batches -> train: {len(train_loader)} | val: {len(val_loader)} | test: {len(test_loader)}")

Batches -> train: 212 | val: 41 | test: 41


## 8 · Arquitectura del modelo

RoBERTa clínico preentrenado + una cabeza de clasificación para las 7 categorías BI-RADS (0–6). Se usa la representación del token inicial (`<s>`).

In [8]:
class AuditorRoBERTa(nn.Module):
    def __init__(self, modelo, n_classes=7, dropout=0.5):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(modelo)
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden, n_classes)
    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]   # token <s>
        return self.classifier(self.dropout(cls))

model = AuditorRoBERTa(MODELO, n_classes=7, dropout=0.5).to(DEVICE)
print(f"Modelo cargado en {DEVICE} | hidden_size = {model.encoder.config.hidden_size}")

pytorch_model.bin:   0%|          | 0.00/504M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: PlanTL-GOB-ES/roberta-base-biomedical-clinical-es
Key                       | Status     | 
--------------------------+------------+-
lm_head.decoder.weight    | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.decoder.bias      | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Modelo cargado en mps | hidden_size = 768


## 9 · Configuración del entrenamiento

Mismos hiperparámetros que el auditor BETO v2 (LR 3e-5, 15 épocas, dropout 0.5, pesos de clase), para una comparación justa.

In [9]:
LR = 3e-5
EPOCHS = 15
PATIENCE = 4

criterio = nn.CrossEntropyLoss(weight=peso_tensor)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)
print(f"LR={LR} | Épocas={EPOCHS} | Patience={PATIENCE} | Pérdida: CrossEntropy ponderada")

LR=3e-05 | Épocas=15 | Patience=4 | Pérdida: CrossEntropy ponderada


## 10 · Entrenamiento

Guarda el mejor modelo según el F1-macro de validación.

In [10]:
def correr_epoca(loader, entrenar=True):
    model.train() if entrenar else model.eval()
    preds, reales, perdida_total = [], [], 0.0
    with torch.set_grad_enabled(entrenar):
        for batch in loader:
            ids = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)
            if entrenar: optimizer.zero_grad()
            logits = model(ids, mask)
            loss = criterio(logits, labels)
            if entrenar:
                loss.backward(); optimizer.step(); scheduler.step()
            perdida_total += loss.item()
            preds.extend(logits.argmax(1).cpu().numpy())
            reales.extend(labels.cpu().numpy())
    f1 = f1_score(reales, preds, average='macro', zero_division=0)
    return perdida_total/len(loader), f1

import os
os.makedirs('../results', exist_ok=True)
RUTA = '../results/mejor_auditor_roberta.pt'

print("🏋️  Entrenando auditor (RoBERTa clínico)")
print("Época | Train Loss | Val Loss | Train F1 | Val F1")
mejor_f1, sin_mejora = 0.0, 0
for ep in range(1, EPOCHS+1):
    tl, tf1 = correr_epoca(train_loader, True)
    vl, vf1 = correr_epoca(val_loader, False)
    marca = ""
    if vf1 > mejor_f1:
        mejor_f1, sin_mejora = vf1, 0
        torch.save(model.state_dict(), RUTA); marca = "  <- mejor"
    else:
        sin_mejora += 1
    print(f"  {ep:2d}  |  {tl:.4f}  |  {vl:.4f}  |  {tf1:.4f}  |  {vf1:.4f}{marca}")
    if sin_mejora >= PATIENCE:
        print(f"Early stopping en época {ep}"); break
print(f"🏆 Mejor Val F1 = {mejor_f1:.4f}")

🏋️  Entrenando auditor (RoBERTa clínico)
Época | Train Loss | Val Loss | Train F1 | Val F1


model.safetensors:   0%|          | 0.00/504M [00:00<?, ?B/s]

   1  |  1.4718  |  1.0184  |  0.2103  |  0.2853  <- mejor
   2  |  0.8242  |  0.9115  |  0.5044  |  0.3712  <- mejor
   3  |  0.4689  |  0.9254  |  0.7418  |  0.4368  <- mejor
   4  |  0.3063  |  1.1959  |  0.8058  |  0.3901
   5  |  0.2161  |  1.0302  |  0.8771  |  0.4222
   6  |  0.1508  |  1.1593  |  0.9167  |  0.4269
   7  |  0.1181  |  1.1788  |  0.9303  |  0.4617  <- mejor
   8  |  0.1351  |  1.1500  |  0.9235  |  0.4661  <- mejor
   9  |  0.0834  |  1.2824  |  0.9528  |  0.4496
  10  |  0.0795  |  1.2938  |  0.9611  |  0.4497
  11  |  0.0700  |  1.3488  |  0.9668  |  0.4780  <- mejor
  12  |  0.0606  |  1.3590  |  0.9724  |  0.4635
  13  |  0.0542  |  1.4303  |  0.9717  |  0.4495
  14  |  0.0482  |  1.4661  |  0.9767  |  0.4397
  15  |  0.0546  |  1.4769  |  0.9783  |  0.4401
Early stopping en época 15
🏆 Mejor Val F1 = 0.4780


## 11 · Evaluación en test

In [11]:
model.load_state_dict(torch.load(RUTA, map_location=DEVICE))
model.eval()
preds, reales = [], []
with torch.no_grad():
    for batch in test_loader:
        ids = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        logits = model(ids, mask)
        preds.extend(logits.argmax(1).cpu().numpy())
        reales.extend(batch['labels'].numpy())

acc = (np.array(preds) == np.array(reales)).mean()
f1m = f1_score(reales, preds, average='macro', zero_division=0)
print("="*55)
print("📊 AUDITOR RoBERTa CLÍNICO — TEST SET")
print("="*55)
print(f"  Accuracy:         {acc:.4f} ({acc*100:.2f}%)")
print(f"  F1-Score (macro): {f1m:.4f}")
print("="*55)
print("\nReporte por clase:")
print(classification_report(reales, preds,
      target_names=[f'BI-RADS {i}' for i in range(7)], zero_division=0))

📊 AUDITOR RoBERTa CLÍNICO — TEST SET
  Accuracy:         0.9006 (90.06%)
  F1-Score (macro): 0.5871

Reporte por clase:
              precision    recall  f1-score   support

   BI-RADS 0       0.81      0.84      0.83       145
   BI-RADS 1       0.93      0.98      0.95        89
   BI-RADS 2       0.98      0.93      0.95       396
   BI-RADS 3       0.29      0.54      0.38        13
   BI-RADS 4       0.50      0.50      0.50         8
   BI-RADS 5       0.50      0.50      0.50         2
   BI-RADS 6       0.00      0.00      0.00         1

    accuracy                           0.90       654
   macro avg       0.57      0.61      0.59       654
weighted avg       0.91      0.90      0.91       654



## 12 · Comparación con BETO

In [12]:
print("Comparación (F1-macro en test, sin fuga de datos):")
print(f"   Auditor BETO (español general):   0.4773")
print(f"   Auditor RoBERTa clínico:          {f1m:.4f}")
delta = f1m - 0.4773
print(f"   Diferencia:                       {delta:+.4f}")
print("\nNota: comparación válida (mismo split, mismo balanceo, semilla 42).")

Comparación (F1-macro en test, sin fuga de datos):
   Auditor BETO (español general):   0.4773
   Auditor RoBERTa clínico:          0.5871
   Diferencia:                       +0.1098

Nota: comparación válida (mismo split, mismo balanceo, semilla 42).
